# Customer Lifetime Value Prediction

**Project Type:** Marketing Analytics | Digital Analytics | Data Science | Predictive Analytics

This project predicts customer 12-month future CLV using ecommerce, web analytics, engagement, acquisition, and retention features.

## Business Objective

Identify which customers are likely to generate the highest future value so that marketing and CRM teams can optimize retention, personalization, acquisition quality, and budget allocation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)


## 1. Load Data


In [ ]:
customers = pd.read_csv('../data/customer_clv_dataset.csv')
transactions = pd.read_csv('../data/transactions.csv')

print('Customer dataset:', customers.shape)
print('Transaction dataset:', transactions.shape)
customers.head()


## 2. Exploratory Data Analysis


In [ ]:
customers.describe().T


In [ ]:
customers['acquisition_channel'].value_counts().plot(kind='bar', figsize=(10,5))
plt.title('Customers by Acquisition Channel')
plt.xlabel('Acquisition Channel')
plt.ylabel('Customers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
customers['future_clv_12m_gbp'].plot(kind='hist', bins=40, figsize=(8,5))
plt.title('Distribution of Future 12-Month CLV')
plt.xlabel('Future CLV GBP')
plt.ylabel('Customers')
plt.tight_layout()
plt.show()


## 3. Feature Selection

The target variable is `future_clv_12m_gbp`. The synthetic seed segment is excluded because it is only used to generate realistic behavioural patterns.


In [ ]:
features = [
    'country', 'primary_device', 'acquisition_channel',
    'tenure_days', 'recency_days', 'frequency_12m', 'avg_order_value_gbp',
    'historical_revenue_12m_gbp', 'gross_margin_rate', 'historical_margin_12m_gbp',
    'discount_usage_rate', 'return_rate', 'sessions_90d', 'product_views_90d',
    'cart_additions_90d', 'email_open_rate', 'support_tickets_12m', 'churn_risk_score'
]

target = 'future_clv_12m_gbp'

X = customers[features]
y = customers[target]

X.head()


## 4. Build Machine Learning Pipeline

The pipeline handles categorical variables using one-hot encoding and trains a Random Forest Regressor for CLV prediction.


In [ ]:
categorical = ['country', 'primary_device', 'acquisition_channel']
numeric = [c for c in features if c not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', 'passthrough', numeric)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=120,
        random_state=42,
        max_depth=12,
        min_samples_leaf=8,
        n_jobs=-1
    ))
])


## 5. Train/Test Split and Model Training


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Training rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])


## 6. Model Evaluation


In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'MAE: £{mae:,.2f}')
print(f'RMSE: £{rmse:,.2f}')
print(f'R²: {r2:.4f}')


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, predictions, alpha=0.35, s=14)
plt.title('Actual vs Predicted 12-Month CLV')
plt.xlabel('Actual Future CLV GBP')
plt.ylabel('Predicted Future CLV GBP')
plt.tight_layout()
plt.show()


## 7. Predict CLV for All Customers


In [ ]:
customers['predicted_clv_12m_gbp'] = model.predict(X).round(2)
customers['clv_tier'] = pd.qcut(
    customers['predicted_clv_12m_gbp'],
    q=4,
    labels=['Low CLV', 'Medium CLV', 'High CLV', 'Top CLV']
)

customers[['customer_id', 'predicted_clv_12m_gbp', 'clv_tier']].head()


## 8. CLV Tier Profiling


In [ ]:
tier_summary = customers.groupby('clv_tier', observed=False).agg(
    customers=('customer_id', 'count'),
    avg_predicted_clv_gbp=('predicted_clv_12m_gbp', 'mean'),
    avg_actual_future_clv_gbp=('future_clv_12m_gbp', 'mean'),
    avg_frequency_12m=('frequency_12m', 'mean'),
    avg_order_value_gbp=('avg_order_value_gbp', 'mean'),
    avg_recency_days=('recency_days', 'mean'),
    avg_churn_risk=('churn_risk_score', 'mean'),
    avg_discount_usage=('discount_usage_rate', 'mean')
).round(2).reset_index()

tier_summary


In [ ]:
tier_summary.plot(x='clv_tier', y='avg_predicted_clv_gbp', kind='bar', figsize=(8,5), legend=False)
plt.title('Average Predicted CLV by Tier')
plt.xlabel('CLV Tier')
plt.ylabel('Average Predicted CLV GBP')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Feature Importance


In [ ]:
rf = model.named_steps['model']
ohe = model.named_steps['preprocessor'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(categorical))
feature_names = cat_names + numeric

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.head(15)


In [ ]:
feature_importance.head(12).sort_values('importance').plot(
    x='feature', y='importance', kind='barh', figsize=(10,6), legend=False
)
plt.title('Top Feature Importance for CLV Prediction')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 10. Business Strategy by CLV Tier

### Top CLV Customers
- Highest predicted future value
- Strong retention priority
- Strategy: VIP loyalty rewards, premium recommendations, early access, personalized CRM journeys

### High CLV Customers
- Strong growth and repeat-purchase potential
- Strategy: cross-sell, bundles, product recommendations, membership incentives

### Medium CLV Customers
- Moderate value with room for growth
- Strategy: nurture campaigns, basket recovery, targeted offers, content personalization

### Low CLV Customers
- Lower future value or higher churn risk
- Strategy: low-cost automation, suppress expensive paid retargeting, test win-back only where margin-positive

## Marketing Analytics Use Cases

- Improve customer retention
- Prioritize CRM campaigns
- Build lookalike audiences from high-CLV customers
- Reduce wasted discounting
- Improve acquisition channel quality
- Support budget allocation by predicted customer value


## 11. Save Outputs


In [ ]:
customers.to_csv('../reports/customer_clv_predictions_from_notebook.csv', index=False)
tier_summary.to_csv('../reports/clv_tier_summary_from_notebook.csv', index=False)
feature_importance.to_csv('../reports/feature_importance_from_notebook.csv', index=False)

print('Outputs saved successfully.')
